In [1]:
import re
import json
from pathlib import Path
from typing import List, Dict, Any
from collections import defaultdict

def pair_folders_by_pattern(
    root_dir: str | Path,
    pattern: str = r"[A-Za-z]*[Kk][A-Za-z]*[-_]?\d{1,3}[-_]\d{1,3}(?:[-_]\d{1,3}){0,2}(?:[-_][A-Za-z]*[Kk][A-Za-z]*[-_]?\d{1,3}[-_]\d{1,3}(?:[-_]\d{1,3}){0,2})?",
    output_json: str | Path | None = None,
    encoding: str = "utf-8"
) -> Dict[str, Any]:
    """
    扫描 root_dir 下的所有子文件夹名，按照指定正则提取标识符进行分组配对。

    参数:
        root_dir     : 要扫描的根目录路径
        pattern      : 用于提取路桩号的正则表达式，兼容二段式、三段式、四段式、五段式、六段式以及两种连接符
        output_json  : 结果保存的 JSON 文件路径。如果为 None 则只返回 dict，不写文件
        encoding     : JSON 文件编码，默认 utf-8
    
    返回:
        {
            "paired": [
                {"key": "K526-010-015","folders": ["...", "..."]},
                ...
            ],
            "unmatched": ["..."]   # 无法配对的文件夹名
        }
    """
    root_path = Path(root_dir).expanduser().resolve()
    if not root_path.is_dir():
        raise FileNotFoundError(f"目录不存在: {root_path}")

    regex = re.compile(pattern)
    groups: Dict[str, List[str]] = defaultdict(list)
    unmatched: List[str] = []

    for item in root_path.iterdir():
        if not item.is_dir():
            continue
        folder_name = item.name
        # 尝试在文件夹名中任意位置找到符合规则的路桩号
        match = regex.search(folder_name)
        if match:
            key = match.group(0)
            groups[key].append(folder_name)
        else:
            unmatched.append(folder_name)

    # 构造字典
    result = {
        "paired": [
            {"key": key, "folders": sorted(folders)}   # 排序让结果更可读
            for key, folders in groups.items()
            if len(folders) >= 2  # 只有 >=2 个才算配对成功
        ],
        "unmatched": sorted(unmatched + [
            folder for key, folders in groups.items()
            if len(folders) == 1
            for folder in folders
        ])
    }

    if output_json:
        output_path = Path(output_json)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        with output_path.open("w", encoding=encoding) as f:
            json.dump(result, f, ensure_ascii=False, indent=2)
        print(f"{root_path.name}的配对结果已保存至: {output_path}")

    return result

DATA_ROOT = r"/home/gary/CloudPointProcessing/20260214学校操场实验/"

result = pair_folders_by_pattern(
    root_dir=DATA_ROOT,
    output_json=r"/home/gary/point-cloud-3d/output/可配对差分的点云名称/20260214学校操场实验.json"
)
# print(json.dumps(result, ensure_ascii=False, indent=2))

20260214学校操场实验的配对结果已保存至: /home/gary/point-cloud-3d/output/可配对差分的点云名称/20260214学校操场实验.json
